# red neuronal para mnist

este proyecto implementa una red neuronal feedforward usando pytorch para clasificar digitos del dataset mnist.

## objetivos
- cargar dataset mnist
- dividir en train/validation/test
- entrenar al menos 3 modelos diferentes
- evaluar en conjunto de test

## importar librerias

solo se permiten pytorch y librerias auxiliares como matplotlib y tqdm

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, SubsetRandomSampler

import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
from tqdm import tqdm

print(f"pytorch version: {torch.__version__}")
print(f"torchvision version: {torchvision.__version__}")

## configuracion inicial

In [ ]:
# configurar device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"usando device: {device}")

# semilla para reproducibilidad
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed(42)

print("configuracion lista")

## verificar instalacion

comprobar que pytorch funciona correctamente

In [ ]:
# crear tensor de prueba
x = torch.randn(3, 3)
print("tensor de prueba:")
print(x)
print(f"shape: {x.shape}")

# mover a device
x = x.to(device)
print(f"tensor en {x.device}")

print("\npytorch funcionando correctamente!")

## cargar dataset mnist

descargar y preparar el dataset mnist con las transformaciones basicas

In [ ]:
# transformaciones para el dataset
# normalizar con media y desviacion estandar de mnist
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

# descargar dataset
train_dataset = torchvision.datasets.MNIST(root='./data', 
                                         train=True, 
                                         transform=transform, 
                                         download=True)

test_dataset = torchvision.datasets.MNIST(root='./data', 
                                        train=False, 
                                        transform=transform, 
                                        download=True)

print(f"dataset train: {len(train_dataset)} muestras")
print(f"dataset test: {len(test_dataset)} muestras")

## dividir train en entrenamiento y validacion

segun la actividad: 50,000 para entrenamiento, 10,000 para validacion

In [ ]:
# indices para dividir el dataset
train_size = 50000
val_size = 10000

# crear indices
indices = list(range(len(train_dataset)))
train_indices = indices[:train_size]
val_indices = indices[train_size:train_size + val_size]

print(f"entrenamiento: {len(train_indices)} muestras")
print(f"validacion: {len(val_indices)} muestras")
print(f"test: {len(test_dataset)} muestras")

# crear samplers para los dataloaders
train_sampler = SubsetRandomSampler(train_indices)
val_sampler = SubsetRandomSampler(val_indices)

## crear dataloaders

configurar dataloaders con batch size basico

In [ ]:
# configuracion de dataloaders
batch_size = 64

train_loader = DataLoader(train_dataset, 
                         batch_size=batch_size, 
                         sampler=train_sampler)

val_loader = DataLoader(train_dataset, 
                       batch_size=batch_size, 
                       sampler=val_sampler)

test_loader = DataLoader(test_dataset, 
                        batch_size=batch_size, 
                        shuffle=False)

print(f"train_loader: {len(train_loader)} batches")
print(f"val_loader: {len(val_loader)} batches")
print(f"test_loader: {len(test_loader)} batches")

## visualizar algunas muestras

mostrar ejemplos del dataset para verificar que esta bien cargado

In [ ]:
# obtener un batch de entrenamiento
dataiter = iter(train_loader)
images, labels = next(dataiter)

print(f"batch shape: {images.shape}")
print(f"labels shape: {labels.shape}")
print(f"labels: {labels[:10]}")

# visualizar primeras 8 imagenes
plt.figure(figsize=(12, 6))
for i in range(8):
    plt.subplot(2, 4, i + 1)
    # denormalizar para visualizar
    img = images[i].squeeze() * 0.3081 + 0.1307
    plt.imshow(img, cmap='gray')
    plt.title(f'digito: {labels[i].item()}')
    plt.axis('off')

plt.tight_layout()
plt.show()

print("\ndataset mnist cargado correctamente!")

## definir red neuronal

crear red neuronal feedforward usando solo nn.Linear y nn.ReLU como especifica la actividad

In [ ]:
class MNISTNet(nn.Module):
    def __init__(self, hidden_size1=128, hidden_size2=64):
        super(MNISTNet, self).__init__()
        # capa de entrada: 28*28 = 784 pixeles
        self.fc1 = nn.Linear(784, hidden_size1)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        # capa de salida: 10 clases (digitos 0-9)
        self.fc3 = nn.Linear(hidden_size2, 10)
        
    def forward(self, x):
        # aplanar la imagen 28x28 a vector 784
        x = x.view(x.size(0), -1)
        # pasar por las capas
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        return x

# crear modelo
model = MNISTNet()
model = model.to(device)

print(f"modelo creado: {model}")
print(f"modelo en device: {next(model.parameters()).device}")

## contar parametros del modelo

funcion simple para contar cuantos parametros tiene el modelo

In [ ]:
def contar_parametros(model):
    total_params = 0
    for param in model.parameters():
        if param.requires_grad:
            total_params += param.numel()
    return total_params

# contar parametros del modelo
total_params = contar_parametros(model)
print(f"total de parametros: {total_params}")

# mostrar arquitectura detallada
print("\ndetalles de la arquitectura:")
for name, layer in model.named_modules():
    if isinstance(layer, (nn.Linear, nn.ReLU)):
        print(f"{name}: {layer}")

print(f"\nforma de entrada esperada: [batch_size, 1, 28, 28]")
print(f"forma de salida: [batch_size, 10]")

## probar el modelo

hacer una prueba rapida para verificar que el modelo funciona

In [ ]:
# obtener un batch para probar
test_images, test_labels = next(iter(train_loader))
test_images = test_images.to(device)

# pasar por el modelo
model.eval()
with torch.no_grad():
    output = model(test_images)

print(f"entrada: {test_images.shape}")
print(f"salida: {output.shape}")
print(f"logits para primera imagen: {output[0]}")
print(f"prediccion (argmax): {torch.argmax(output[0]).item()}")
print(f"etiqueta real: {test_labels[0].item()}")

print("\nmodelo definido correctamente!")

## configurar entrenamiento

definir funcion de perdida, optimizador y funciones de evaluacion

In [ ]:
# funcion de perdida para clasificacion multiclase
criterion = nn.CrossEntropyLoss()

# optimizador - empezamos con SGD basico
learning_rate = 0.01
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

print(f"funcion de perdida: {criterion}")
print(f"optimizador: {optimizer}")
print(f"learning rate: {learning_rate}")

## funciones de evaluacion

implementar funciones para calcular accuracy sin usar sklearn

In [ ]:
def calcular_accuracy(model, dataloader, device):
    """
    calcular accuracy del modelo en un dataloader
    """
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    
    accuracy = 100 * correct / total
    return accuracy

# probar la funcion
test_accuracy = calcular_accuracy(model, val_loader, device)
print(f"accuracy inicial (random): {test_accuracy:.2f}%")

## funcion para entrenar una epoca

implementar el loop de entrenamiento para una epoca

In [ ]:
def entrenar_una_epoca(model, train_loader, criterion, optimizer, device):
    """
    entrenar el modelo por una epoca
    """
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # usar tqdm para mostrar progreso
    train_bar = tqdm(train_loader, desc='entrenando')
    
    for batch_idx, (images, labels) in enumerate(train_bar):
        # mover datos al device
        images, labels = images.to(device), labels.to(device)
        
        # limpiar gradientes
        optimizer.zero_grad()
        
        # forward pass
        outputs = model(images)
        loss = criterion(outputs, labels)
        
        # backward pass
        loss.backward()
        optimizer.step()
        
        # estadisticas
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()
        
        # actualizar barra de progreso
        if batch_idx % 100 == 0:
            accuracy = 100 * correct / total
            train_bar.set_postfix({\n                'loss': f'{running_loss/(batch_idx+1):.3f}',\n                'acc': f'{accuracy:.2f}%'\n            })\n    \n    # estadisticas finales de la epoca\n    epoch_loss = running_loss / len(train_loader)\n    epoch_accuracy = 100 * correct / total\n    \n    return epoch_loss, epoch_accuracy

## funcion de entrenamiento completo

funcion principal para entrenar el modelo por varias epocas

In [ ]:
def entrenar_modelo(model, train_loader, val_loader, criterion, optimizer, num_epochs, device):
    """
    entrenar el modelo por varias epocas y guardar metricas
    """
    train_losses = []
    train_accuracies = []
    val_accuracies = []
    
    print(f"iniciando entrenamiento por {num_epochs} epocas...")
    print("-" * 50)
    
    for epoch in range(num_epochs):
        print(f"\\nepoca {epoch+1}/{num_epochs}")
        
        # entrenar una epoca
        train_loss, train_acc = entrenar_una_epoca(model, train_loader, criterion, optimizer, device)
        
        # evaluar en validacion
        val_acc = calcular_accuracy(model, val_loader, device)
        
        # guardar metricas
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_accuracies.append(val_acc)
        
        # mostrar resultados de la epoca
        print(f"train loss: {train_loss:.4f} | train acc: {train_acc:.2f}% | val acc: {val_acc:.2f}%")
    
    print("\\nentrenamiento completado!")
    return train_losses, train_accuracies, val_accuracies

print("funciones de entrenamiento configuradas correctamente!")

# entrenamiento de modelos

## modelo 1: configuracion basica

entrenar el primer modelo con configuracion basica (SGD, lr=0.01, arquitectura 784->128->64->10)

In [ ]:
# reinicializar el modelo para empezar desde cero
model1 = MNISTNet(hidden_size1=128, hidden_size2=64).to(device)
criterion1 = nn.CrossEntropyLoss()
optimizer1 = optim.SGD(model1.parameters(), lr=0.01)

print("modelo 1 - configuracion:")
print(f"  arquitectura: {784} -> {128} -> {64} -> {10}")
print(f"  optimizador: SGD")
print(f"  learning rate: {0.01}")
print(f"  parametros totales: {contar_parametros(model1)}")

# entrenar modelo 1
num_epochs = 10

print(f"\\niniciando entrenamiento modelo 1...")
train_losses1, train_accs1, val_accs1 = entrenar_modelo(\n    model1, train_loader, val_loader, criterion1, optimizer1, num_epochs, device\n)

## resultados modelo 1

mostrar metricas y graficas del primer modelo

In [ ]:
# mostrar resultados finales del modelo 1
print("\\nresultados finales modelo 1:")
print(f"  accuracy final entrenamiento: {train_accs1[-1]:.2f}%")
print(f"  accuracy final validacion: {val_accs1[-1]:.2f}%")
print(f"  loss final: {train_losses1[-1]:.4f}")

# graficar resultados
plt.figure(figsize=(15, 5))

# loss
plt.subplot(1, 3, 1)
plt.plot(train_losses1, label='train loss')
plt.title('loss modelo 1')
plt.xlabel('epoca')
plt.ylabel('loss')
plt.legend()
plt.grid(True)

# accuracy entrenamiento
plt.subplot(1, 3, 2)
plt.plot(train_accs1, label='train accuracy')
plt.title('accuracy entrenamiento modelo 1')
plt.xlabel('epoca')
plt.ylabel('accuracy (%)')
plt.legend()
plt.grid(True)

# accuracy validacion
plt.subplot(1, 3, 3)
plt.plot(val_accs1, label='validation accuracy', color='orange')
plt.title('accuracy validacion modelo 1')
plt.xlabel('epoca')
plt.ylabel('accuracy (%)')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

# guardar metricas para comparacion posterior
modelo1_resultados = {
    'nombre': 'modelo_basico',
    'arquitectura': '784->128->64->10',
    'optimizador': 'SGD',
    'lr': 0.01,
    'epochs': num_epochs,
    'val_accuracy': val_accs1[-1],
    'train_accuracy': train_accs1[-1],
    'final_loss': train_losses1[-1]
}

print(f"\\nmodelo 1 entrenado y guardado!")

## modelo 2: arquitectura mas simple

probar con menos neuronas - 784->256->10 (solo una capa oculta)

In [ ]:
# definir modelo 2 con arquitectura diferente
class MNISTNet2(nn.Module):
    def __init__(self):
        super(MNISTNet2, self).__init__()
        # arquitectura mas simple: solo una capa oculta
        self.fc1 = nn.Linear(784, 256)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(256, 10)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        return x

# crear y entrenar modelo 2
model2 = MNISTNet2().to(device)
criterion2 = nn.CrossEntropyLoss()
optimizer2 = optim.SGD(model2.parameters(), lr=0.01)

print("modelo 2 - configuracion:")
print(f"  arquitectura: {784} -> {256} -> {10}")
print(f"  optimizador: SGD")
print(f"  learning rate: {0.01}")
print(f"  parametros totales: {contar_parametros(model2)}")

print(f"\\niniciando entrenamiento modelo 2...")
train_losses2, train_accs2, val_accs2 = entrenar_modelo(\n    model2, train_loader, val_loader, criterion2, optimizer2, num_epochs, device\n)

## modelo 3: arquitectura mas profunda

probar con mas capas - 784->128->64->32->10

In [ ]:
# definir modelo 3 con mas capas
class MNISTNet3(nn.Module):
    def __init__(self):
        super(MNISTNet3, self).__init__()
        # arquitectura mas profunda
        self.fc1 = nn.Linear(784, 128)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(128, 64)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(64, 32)
        self.relu3 = nn.ReLU()
        self.fc4 = nn.Linear(32, 10)
        
    def forward(self, x):
        x = x.view(x.size(0), -1)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.fc3(x)
        x = self.relu3(x)
        x = self.fc4(x)
        return x

# crear y entrenar modelo 3
model3 = MNISTNet3().to(device)
criterion3 = nn.CrossEntropyLoss()
optimizer3 = optim.SGD(model3.parameters(), lr=0.01)

print("modelo 3 - configuracion:")
print(f"  arquitectura: {784} -> {128} -> {64} -> {32} -> {10}")
print(f"  optimizador: SGD")
print(f"  learning rate: {0.01}")
print(f"  parametros totales: {contar_parametros(model3)}")

print(f"\\niniciando entrenamiento modelo 3...")
train_losses3, train_accs3, val_accs3 = entrenar_modelo(\n    model3, train_loader, val_loader, criterion3, optimizer3, num_epochs, device\n)

## guardar resultados de todos los modelos

almacenar metricas de los 3 modelos para comparacion

In [ ]:
# guardar resultados de todos los modelos
modelo2_resultados = {
    'nombre': 'modelo_simple',
    'arquitectura': '784->256->10',
    'optimizador': 'SGD',
    'lr': 0.01,
    'epochs': num_epochs,
    'val_accuracy': val_accs2[-1],
    'train_accuracy': train_accs2[-1],
    'final_loss': train_losses2[-1]
}

modelo3_resultados = {
    'nombre': 'modelo_profundo',
    'arquitectura': '784->128->64->32->10',
    'optimizador': 'SGD',
    'lr': 0.01,
    'epochs': num_epochs,
    'val_accuracy': val_accs3[-1],
    'train_accuracy': train_accs3[-1],
    'final_loss': train_losses3[-1]
}

# lista con todos los modelos
todos_los_modelos = [modelo1_resultados, modelo2_resultados, modelo3_resultados]

print("\\ncomparacion de arquitecturas:")
print("-" * 60)
for modelo in todos_los_modelos:
    print(f"{modelo['nombre']}: {modelo['arquitectura']}")
    print(f"  val accuracy: {modelo['val_accuracy']:.2f}%")
    print(f"  train accuracy: {modelo['train_accuracy']:.2f}%")
    print(f"  parametros: {contar_parametros(eval(modelo['nombre'].replace('modelo_', 'model').replace('basico', '1').replace('simple', '2').replace('profundo', '3')))}")
    print()